# CLIP 3D Spatial Relationships

## init

In [10]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataloader import create_dataloader, SpatialDataset
import matplotlib.pyplot as plt
from collections import defaultdict

CLIP_MODEL_NAME = "openai/clip-vit-base-patch16"

device = "cuda" if torch.cuda.is_available() else "cpu"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## clip model

In [2]:
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

## data & feature extraction

data filepath format: `<c/nc>_<per_train>_<c/nc>_<per_valid>`

1. determine which predicates to test
1. for each sample within the filtered predicates, get the following clip embeddings:
    * full image embedding
    * subject text emb
    * objext text emb
    * predicate text emb
    * depth image embedding (in case we end up using it)
1. create individual datasets for each predicate, each with the following keys:
    * image_emb
    * subject_emb
    * object_emb
    * predicate_emb
    * depth_emb
    * weight (weight from sample, in case we end up using it)
1. for attention map analysis: probably fine to just analyze a random sample(s) from each predicate, don't need to track attention maps for each sample 

* experiment with cropped vs uncropped images?
* experiment with holding out certain objects for testing? 


**sample keys**
* `subject`:
    * object name
    * object idx
    * bounding box
    * camera pose tensor (t) (?) 
* `object`:
    * same as `subject`
* `predicate`:
    * predicate name
    * predicate idx
    * bbox
* `label`:
    * True/False, whether predicate holds
* `rgb_source`:
    * data path to original photo, likely path in gcp bucket
* `weight`:
    * ??? weight for correctness of label?
* `img_crop`:
    * rbg img
* `depth_crop`:
    * depth img
* `bbox_mask`:
    * mask for bounding box 

In [ ]:
default_dataset_args = {
    "split": 'train',
    "predicate_dim": 30,
    "object_dim": 67,
    "data_path": 'data/c_0.9_c_0.1.json',
    "load_img": True,
    "data_aug_shift": False,
    "data_aug_color": False,
    "crop": True,
    "norm_data": False,
    "resize_mask": False,
    "trans_vec": [],
}

allowed_predicates = set([
    "in front of", # occlusion, depth
    "behind", # occlusion, depth
    "below", # relative position
    "to the left of (wrt you)", # relative position
    "to the right of (wrt you)", # relative position
    "to the side of", # relative position
    "on", # relative position
    "covering", # occlusion
    "inside", # occlusion
    "near", # depth
    "far from", # depth
])

train_dataset_args = default_dataset_args | {"split": "train"}
# val_dataset_args = default_dataset_args | {"split": "valid"}
# test_dataset_args = default_dataset_args | {"split": "test"}

train_dl, train_predicates, train_objects = create_dataloader(
    **train_dataset_args,
    num_workers=1,
    batch_size=16,
)
# val_dl, val_predicates, val_objects = create_dataloader(
#     **val_dataset_args,
#     num_workers=1,
#     batch_size=16,
# )
# test_dl, test_predicates, test_objects = create_dataloader(
#     **test_dataset_args,
#     num_workers=1,
#     batch_size=16,
# )

20454 relations in train
Percentage of positive labels  in train: 0.5
Percentage of negative labels  in train: 0.5
{'to the side of (wrt you)': (72, 72, 0.5), 'far from': (89, 89, 0.5), 'to the left of': (154, 154, 0.5), 'to the side of': (515, 515, 0.5), 'around': (326, 326, 0.5), 'under': (751, 751, 0.5), 'over': (557, 557, 0.5), 'aligned to': (232, 232, 0.5), 'near': (225, 225, 0.5), 'touching': (276, 276, 0.5), 'behind (wrt you)': (266, 266, 0.5), 'behind': (427, 427, 0.5), 'on': (722, 722, 0.5), 'points towards': (224, 224, 0.5), 'faces away': (495, 495, 0.5), 'faces towards': (454, 454, 0.5), 'to the right of': (161, 161, 0.5), 'to the right of (wrt you)': (477, 477, 0.5), 'in front of (wrt you)': (211, 211, 0.5), 'inside': (213, 213, 0.5), 'below': (466, 466, 0.5), 'points away': (284, 284, 0.5), 'leaning against': (330, 330, 0.5), 'outside': (28, 28, 0.5), 'passing through': (72, 72, 0.5), 'covering': (128, 128, 0.5), 'to the left of (wrt you)': (509, 509, 0.5), 'in': (497, 497

In [ ]:
class PredicateProbeDataset(Dataset):
    def __init__(self, pt_file_path):
        """
        pt_file_path should have the following format:
            <predicate>_<train/val/test>.pt

        Each sample should have the following keys:
            image_emb: rgb image embedding
            subject_emb: text embedding of subject
            object_emb: text embedding of object
            predicate_emb: text embedding of predicate
            depth_emb: depth image embedding
            subject_name: subject text
            object_name: object text
            predicate_name: predicate text
            weight: Rel3d sample weight
        """
        self.samples = torch.load(pt_file_path)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, ix):
        return self.samples[ix]

NameError: name 'Dataset' is not defined

In [ ]:
samples_dict = defaultdict(list)

for split in ("train", "valid", "test"):
    split_dataset_args = default_dataset_args | {"split": split}
    split_data_loader, split_predicates, split_objects = create_dataloader(
        **split_dataset_args,
        num_workers=1,
        batch_size=16,
    )

    for batch in split_data_loader:
        # prepare inputs
        images = [image for image in batch["img_crop"]]
        subject_names = batch["subject"]["name"]
        object_names = batch["object"]["name"]
        predicate_names = batch["predicate"]["name"]
        depth_images = [depth_image.repeat(3, 1, 1) for depth_image in batch["depth_crop"]] # duplicate channel dim
        weights = batch["weight"]
        labels = batch["label"]

        # encode images
        image_inputs = processor(images=images, return_tensors="pt").to(device)
        depth_image_inputs = processor(images=depth_images, return_tensors="pt").to(device)
        with torch.no_grad():
            image_embs = model.get_image_features(**image_inputs)
            depth_image_embs = model.get_image_features(**depth_image_inputs)

        # encode text
        subject_inputs = processor(text=subject_names, return_tensors="pt", padding=True).to(device)
        object_inputs = processor(text=object_names, return_tensors="pt", padding=True).to(device)
        predicate_inputs = processor(text=predicate_names, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            subject_embs = model.get_text_features(**subject_inputs)
            object_embs = model.get_text_features(**object_inputs)
            predicate_embs = model.get_text_features(**predicate_inputs)

        # construct and sort samples
        for i in range(len(images)):
            if predicate_names[i] in allowed_predicates:
                sample = {
                    "image_emb": image_embs[i],
                    "subject_emb": subject_embs[i],
                    "object_emb": object_embs[i],
                    "predicate_emb": predicate_embs[i],
                    "depth_emb": depth_image_embs[i],
                    "subject_name": subject_names[i],
                    "object_name": object_names[i],
                    "predicate_name": predicate_names[i],
                    "weight": weights[i],
                }
                samples_dict[predicate_names[i]].append(sample)

        break
    break

20454 relations in train
Percentage of positive labels  in train: 0.5
Percentage of negative labels  in train: 0.5
{'to the side of (wrt you)': (72, 72, 0.5), 'far from': (89, 89, 0.5), 'to the left of': (154, 154, 0.5), 'to the side of': (515, 515, 0.5), 'around': (326, 326, 0.5), 'under': (751, 751, 0.5), 'over': (557, 557, 0.5), 'aligned to': (232, 232, 0.5), 'near': (225, 225, 0.5), 'touching': (276, 276, 0.5), 'behind (wrt you)': (266, 266, 0.5), 'behind': (427, 427, 0.5), 'on': (722, 722, 0.5), 'points towards': (224, 224, 0.5), 'faces away': (495, 495, 0.5), 'faces towards': (454, 454, 0.5), 'to the right of': (161, 161, 0.5), 'to the right of (wrt you)': (477, 477, 0.5), 'in front of (wrt you)': (211, 211, 0.5), 'inside': (213, 213, 0.5), 'below': (466, 466, 0.5), 'points away': (284, 284, 0.5), 'leaning against': (330, 330, 0.5), 'outside': (28, 28, 0.5), 'passing through': (72, 72, 0.5), 'covering': (128, 128, 0.5), 'to the left of (wrt you)': (509, 509, 0.5), 'in': (497, 497

Python(62904) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [20]:
samples_dict.keys()

dict_keys(['to the left of (wrt you)', 'below', 'to the right of (wrt you)', 'to the side of', 'behind', 'on'])

In [36]:
for x in train_dl:
    sample = x
    break

sample.keys()

Python(72397) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


dict_keys(['subject', 'object', 'predicate', 'label', 'rgb_source', 'weight', 'img_crop', 'depth_crop', 'bbox_mask'])

In [36]:
image = Image.open("data/apple.jpg")
text = "a red gala apple"

# preprocess
inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

# encode
with torch.no_grad():
    outputs = model(**inputs)
    image_emb = outputs.image_embeds
    text_emb = outputs.text_embeds

In [37]:
outputs.keys()

odict_keys(['logits_per_image', 'logits_per_text', 'text_embeds', 'image_embeds', 'text_model_output', 'vision_model_output'])

## attention maps

* use vision model outputs to access attention maps (`outputs["vision_model_output"]`)
* outputs keys: `odict_keys(['logits_per_image', 'logits_per_text', 'text_embeds', 'image_embeds', 'text_model_output', 'vision_model_output'])`

1. for each predicate, analyze attention maps of a few examples
1. try to draw some comparisons/conclusions among samples of the same predicate
1. try to draw some comparisons/conclusions between samples of different predicates 
1. do insights from attention map analysis correspond to binary classifier performance at all?
1. is the `bbox_mask` associated with each sample helpful in attention mask analysis? for masking irrelevant areas of the image/attention maps? 

In [21]:
# model.config.output_attentions = True

with torch.no_grad():
    vision_outputs = model.vision_model(
        pixel_values=inputs["pixel_values"], output_attentions=True
    )
    text_outputs = model.text_model(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], output_attentions=True
    )

# Example: attention from 3rd layer of ViT, head 0
attn_map = vision_outputs.attentions[2][0, 0]  # shape: (num_tokens, num_tokens)
attn_map

tensor([[9.2208e-01, 6.1286e-05, 5.0012e-05,  ..., 2.4218e-04, 2.7706e-04,
         5.0046e-04],
        [4.5923e-02, 7.6754e-02, 6.2790e-02,  ..., 8.8701e-05, 1.3569e-04,
         6.3760e-05],
        [5.2030e-02, 6.1412e-02, 5.0170e-02,  ..., 7.7510e-05, 1.1523e-04,
         5.0666e-05],
        ...,
        [2.0385e-01, 4.8171e-04, 3.4117e-04,  ..., 9.0197e-02, 1.0725e-01,
         2.8560e-02],
        [2.1190e-01, 6.0444e-04, 4.0130e-04,  ..., 9.6132e-02, 1.1625e-01,
         3.6867e-02],
        [2.2307e-01, 5.2531e-04, 3.6921e-04,  ..., 6.7901e-02, 9.9032e-02,
         1.4313e-01]])

In [22]:
vision_outputs.attentions[2].shape

torch.Size([1, 12, 197, 197])

In [ ]:
image = Image.open("data/apple.jpg")
text = "a red gala apple"

# preprocess
inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

# encode
with torch.no_grad():
    outputs = model(**inputs)
    image_emb = outputs.image_embeds
    text_emb = outputs.text_embeds

outputs.vision_model_output # givesn the vision_outputs above

## test

In [32]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

# Load model and processor
test_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
test_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
test_model.eval()

# Load your image
image = Image.open("vis_10004.jpg")

# Define candidate class labels
labels = ["a photo of a couch", "a photo of a table", "a photo of a pear"]

# Preprocess
inputs = test_processor(text=labels, images=image, return_tensors="pt", padding=True)

# Forward pass
with torch.no_grad():
    outputs = test_model(**inputs)
    logits_per_image = outputs.logits_per_image  # shape: [1, len(labels)]
    probs = logits_per_image.softmax(dim=1)

# Print result
for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

# Predicted label
pred = labels[probs.argmax()]
print(f"\nPrediction: {pred}")

a photo of a couch: 0.9706
a photo of a table: 0.0294
a photo of a pear: 0.0000

Prediction: a photo of a couch
